In [1]:
# Save total mortality by country for all ensemble members

In [2]:
import os
import xarray as xr
import numpy as np
from utils.utils import get_scenario_config

In [3]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

in_file = "GBD_Country_Masks_0.10.nc"
in_path = os.path.join(MASK_DIR, in_file)
country_mask = xr.open_dataarray(in_path)

In [ ]:
# === Path config ===
MORTALITY_DIR = "/glade/work/awells/air_quality/CESM/mortality/ozone/"

# Set to whatever scenario you want, function returns error if not recognised
scenario = "SSP245_G6"

config = get_scenario_config(scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

# === Main loop ===
ensembles = []
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # {years.stop - 1} from OSDMA8 calculation
    dates = f"{years.start}-{years.stop - 1}"

    in_file = f"Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(MORTALITY_DIR, in_file)

    if not os.path.exists(in_path):
        print(f"Missing: {in_path}")
        continue
    da = xr.open_dataarray(in_path)

    countries = []
    # Loop over countries and sum the mortality for each country
    for i in range(len(country_mask.country)):
        print(f"Country number {i}")
        mask = country_mask.isel(country=i)
        # mean mortality of country (based on BMR mean)
        M_country = (xr.where(mask == 1,
                              da.sel(quantile="mean"),
                              np.nan)).sum(dim=("lat", "lon"))
        countries.append(M_country.drop_vars("country", errors='ignore'))

    mortality_country = xr.concat(countries,
                                  dim=xr.DataArray(country_mask["country"],
                                                   dims="country",
                                                   name="country"))

    ensembles.append(mortality_country)

countries_ens = xr.concat(ensembles,
                          dim=xr.DataArray(np.arange(1, len(ensembles)+1),
                                           dims="ensemble", name="ensemble"))

countries_ens.attrs["description"] = ("Country level mean Mortality - "
                                      "scripts by A.F. Wells (2025)")
countries_ens.attrs["scenario"] = scenario

out_file = f"Mortality_Country_sum_CESM2_{scenario}_{dates}.nc"
out_path = os.path.join(MORTALITY_DIR, out_file)
print(f"Saving country mortality to {out_path}")
countries_ens.to_netcdf(out_path)

Processing SSP245_G6, Ensemble 01
Country number 0
Country number 1
Country number 2
Country number 3
Country number 4
Country number 5
Country number 6
Country number 7
Country number 8
Country number 9
Country number 10
Country number 11
Country number 12
Country number 13
Country number 14
Country number 15
Country number 16
Country number 17
Country number 18
Country number 19
Country number 20
Country number 21
Country number 22
Country number 23
Country number 24
Country number 25
Country number 26
Country number 27
Country number 28
Country number 29
Country number 30
Country number 31
Country number 32
Country number 33
Country number 34
Country number 35
Country number 36
Country number 37
Country number 38
Country number 39
Country number 40
Country number 41
Country number 42
Country number 43
Country number 44
Country number 45
Country number 46
Country number 47
Country number 48
Country number 49
Country number 50
Country number 51
Country number 52
Country number 53
Coun